# Visualize built smoke tubes

Loads tube JSONs from `data/03_primary/tubes/{split}/` and renders timelines + filmstrips. No YOLO model required.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from smokeynet_adapted.data import (
    find_sequence_dir,
    get_sorted_frames,
    load_tube_record,
)
from smokeynet_adapted.tubes import plot_tube_summary, tube_from_record

In [ ]:
SPLIT = "val"
TUBES_DIR = Path(f"../data/03_primary/tubes/{SPLIT}")
RAW_DIR = Path(f"../data/01_raw/datasets/{SPLIT}")
PER_LABEL = 3  # show this many smoke + this many fp

summary = load_tube_record(TUBES_DIR / "_summary.json")
n_written = summary["tubes_written"]
n_total = summary["total_sequences"]
print(f"split={summary['split']}  written={n_written}/{n_total}")
print(f"by_label={summary['by_label']}  dropped={len(summary['dropped'])}")

In [ ]:
all_files = sorted(p for p in TUBES_DIR.glob("*.json") if p.name != "_summary.json")
by_label: dict[str, list[Path]] = {"smoke": [], "fp": []}
for p in all_files:
    rec = load_tube_record(p)
    by_label[rec["label"]].append(p)
print(f"smoke files: {len(by_label['smoke'])}, fp files: {len(by_label['fp'])}")

selected = by_label["smoke"][:PER_LABEL] + by_label["fp"][:PER_LABEL]
print(f"showing {len(selected)} sequences ({PER_LABEL} smoke + {PER_LABEL} fp)")

In [ ]:
for path in selected:
    record = load_tube_record(path)
    tube = tube_from_record(record)
    seq_dir = find_sequence_dir(RAW_DIR, record["sequence_id"])
    if seq_dir is None:
        print(f"skipping {record['sequence_id']}: source dir not found")
        continue
    frame_paths = get_sorted_frames(seq_dir)
    fig = plot_tube_summary(
        frame_paths,
        [tube],
        num_frames=record["num_frames"],
        tube_labels=[record["label"] == "smoke"],
        title=f"{record['sequence_id']} [{record['label']}]",
    )
    plt.show()